# Exercise 59 — GroupBy

## Concept

`GroupBy` is the SQL `GROUP BY` of pandas: split the rows into buckets, apply an aggregation per bucket, combine.

```python
df.groupby("region")["amount"].sum()           # Series indexed by region
df.groupby("region")["amount"].agg(["sum", "mean", "count"])    # DataFrame
df.groupby(["region", "customer"])["amount"].sum()              # multi-level group
```

### Named aggregations (clearer column names)

```python
df.groupby("region").agg(
    total=("amount", "sum"),
    avg=("amount", "mean"),
    n_orders=("order_id", "count"),
)
# columns: total, avg, n_orders  (instead of 'amount_sum', etc.)
```

### Get a plain DataFrame back

`groupby(...).agg(...)` puts the grouping keys in the **index**. To get them as regular columns, chain `.reset_index()`:

```python
df.groupby("region")["amount"].sum().reset_index()
# columns: region, amount
```

Or pass `as_index=False` to `.groupby()`:

```python
df.groupby("region", as_index=False)["amount"].sum()
```

### Common aggregations

`"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`, `"count"` (non-null), `"size"` (total rows including null), `"nunique"`, `"first"`, `"last"`. Custom: `.agg(lambda s: ...)` — slower but works.

### `count` vs `size`

`count` ignores NaN; `size` doesn't. Important when nulls matter.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "customer": ["Alice", "Bob", "Alice", "Carol", "Bob", "Diana", "Alice", "Carol", "Bob", "Eve"],
    "region":   ["N", "S", "N", "S", "N", "E", "S", "N", "S", "E"],
    "amount":   [120.0, 45.0, 300.0, 89.0, 210.0, 55.0, 430.0, 12.0, np.nan, 750.0],
})
print(df)


## Task 1 — Total by region

Return a DataFrame with columns `["region", "total"]`, one row per region, sorted by `region` ascending.

```python
def totals_by_region(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected:
```
  region   total
0      E   805.0
1      N   642.0
2      S   564.0
```

Note: by default `.sum()` skips NaN, so order_id 9's missing amount doesn't trip you up.

In [ ]:
import pandas as pd

def totals_by_region(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = totals_by_region(df)
assert list(result.columns) == ["region", "total"]
assert result["region"].tolist() == ["E", "N", "S"]
assert result["total"].tolist() == [805.0, 642.0, 564.0]
print("ok")


## Task 2 — Multiple aggregations with named output

Return one row per region with columns:
- `total` — sum of `amount`
- `avg`   — mean of `amount`
- `n_orders` — count of `order_id`
- `n_unique_customers` — distinct count of `customer`

Use `.agg(...)` with named aggregations. Columns must appear in that exact order. Reset the index so `region` is a column.

```python
def region_stats(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

In [ ]:
import pandas as pd

def region_stats(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = region_stats(df).sort_values("region").reset_index(drop=True)
assert list(result.columns) == ["region", "total", "avg", "n_orders", "n_unique_customers"]
row_n = result[result["region"] == "N"].iloc[0]
assert row_n["total"] == 642.0
assert row_n["n_orders"] == 4
assert row_n["n_unique_customers"] == 3   # Alice, Bob, Carol
print("ok")


## Task 3 — Top customer per region

For each region, find the customer with the **highest total `amount`**. Return a DataFrame with columns `["region", "customer", "total"]`, one row per region.

```python
def top_customer_per_region(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Hint: group by `[region, customer]`, sum amounts, then within each region pick the row with max total. One way:
```python
totals = df.groupby(["region", "customer"], as_index=False)["amount"].sum()
totals.sort_values("amount", ascending=False).drop_duplicates("region")
```
Then tidy column names and sort by region.

Expected (sorted by region):
```
  region customer  total
0      E      Eve  750.0
1      N    Alice  420.0
2      S    Alice  430.0
```

In [ ]:
import pandas as pd

def top_customer_per_region(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = top_customer_per_region(df)
assert list(result.columns) == ["region", "customer", "total"]
assert result.set_index("region").loc["E"]["customer"] == "Eve"
assert result.set_index("region").loc["E"]["total"] == 750.0
assert result.set_index("region").loc["N"]["customer"] == "Alice"
assert result.set_index("region").loc["N"]["total"] == 420.0  # 120 + 300
assert result.set_index("region").loc["S"]["customer"] == "Alice"
assert result.set_index("region").loc["S"]["total"] == 430.0  # just order 7
print("ok")


## Task 4 — Custom aggregation: range

For each region, return the **range** (max - min) of the `amount` column. Use a single `.agg(...)` call with a lambda. Return a DataFrame with columns `["region", "amount_range"]`.

```python
def amount_range_per_region(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected (sorted by region):
```
  region   amount_range
0      E         695.0   (750 - 55)
1      N         288.0   (300 - 12)
2      S         385.0   (430 - 45, ignoring NaN)
```

Lambda aggregations are slower than the built-in strings but handle anything. Built-in aggregations are vectorized in C; lambdas run Python per group.

In [ ]:
import pandas as pd

def amount_range_per_region(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = amount_range_per_region(df).sort_values("region").reset_index(drop=True)
assert list(result.columns) == ["region", "amount_range"]
assert result["region"].tolist() == ["E", "N", "S"]
assert result["amount_range"].tolist() == [695.0, 288.0, 385.0]
print("ok")
